# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)


/mnt/e/EE446/lab3/tinyml-arduino/bin/python


In [2]:
# Environment check only. The assignment asks not to install or uninstall
# TensorFlow packages inside this notebook.
import importlib.util

required_packages = [
    "tensorflow",
    "tensorflow_model_optimization",
    "sklearn",
    "numpy",
    "pandas",
]

for package_name in required_packages:
    status = "found" if importlib.util.find_spec(package_name) else "missing"
    print(f"{package_name}: {status}")


tensorflow: found
tensorflow_model_optimization: found
sklearn: found
numpy: found
pandas: found


In [3]:
# If TensorFlow or TF-MOT is reported as missing above, switch JupyterLab to
# the class kernel named Python (tinyml-arduino), then run this notebook again.
# Do not install packages from this notebook.


In [4]:
from pathlib import Path
import contextlib
import io
import os
import warnings

# Suppress TensorFlow converter/status logs so assignment outputs stay readable.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", message="Statistics for quantized inputs were expected.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
import tensorflow_model_optimization as tfmot

tf.get_logger().setLevel("ERROR")

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
to_categorical = tf.keras.utils.to_categorical

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)


2026-05-19 16:04:14.784137: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-19 16:04:14.784582: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-19 16:04:14.786006: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow version: 2.14.1
TF-MOT version: 0.8.0


---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset provided with the assignment. The local `wine.data` file is loaded, the original class labels `1`, `2`, and `3` are mapped to zero-based labels for Keras, and all generated TFLite artifacts are saved in the same HW1 folder.


In [5]:
# Load the Wine dataset from the local assignment files.

def resolve_hw1_dir():
    candidates = [Path.cwd(), Path.cwd() / "HW1", Path.cwd().parent / "HW1"]
    for candidate in candidates:
        if (candidate / "wine.data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find wine.data in the current folder, HW1, or ../HW1.")

HW1_DIR = resolve_hw1_dir()
DATA_PATH = HW1_DIR / "wine.data"

feature_names = [
    "Alcohol",
    "Malic acid",
    "Ash",
    "Alcalinity of ash",
    "Magnesium",
    "Total phenols",
    "Flavanoids",
    "Nonflavanoid phenols",
    "Proanthocyanins",
    "Color intensity",
    "Hue",
    "OD280/OD315 of diluted wines",
    "Proline",
]

columns = ["Class"] + feature_names
df = pd.read_csv(DATA_PATH, header=None, names=columns)

# Convert original labels 1, 2, 3 to Keras-friendly labels 0, 1, 2.
df["Class"] = df["Class"].astype(int) - 1
class_names = [f"Cultivar {i + 1}" for i in sorted(df["Class"].unique())]

num_classes = df["Class"].nunique()
num_features = len(feature_names)

print("Dataset path:", DATA_PATH)
print("Number of samples:", len(df))
print("Number of classes:", num_classes)
print("Number of features:", num_features)
print("Class mapping: original labels 1, 2, 3 -> encoded labels 0, 1, 2")

feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


def artifact_path(filename):
    path = Path(filename)
    return path if path.is_absolute() else HW1_DIR / path


def write_tflite_model(tflite_model, filename):
    path = artifact_path(filename)
    path.write_bytes(tflite_model)
    return path


def file_size_kb(filename):
    return artifact_path(filename).stat().st_size / 1024


def convert_tflite(converter):
    """Convert a model while hiding verbose TensorFlow converter logs."""
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return converter.convert()


results = {}


Dataset path: /mnt/e/EE446/HW1/wine.data
Number of samples: 178
Number of classes: 3
Number of features: 13
Class mapping: original labels 1, 2, 3 -> encoded labels 0, 1, 2

Feature statistics:
                                  min      max        mean         std
Alcohol                        11.03    14.83   13.000618    0.811827
Malic acid                      0.74     5.80    2.336348    1.117146
Ash                             1.36     3.23    2.366517    0.274344
Alcalinity of ash              10.60    30.00   19.494944    3.339564
Magnesium                      70.00   162.00   99.741573   14.282484
Total phenols                   0.98     3.88    2.295112    0.625851
Flavanoids                      0.34     5.08    2.029270    0.998859
Nonflavanoid phenols            0.13     0.66    0.361854    0.124453
Proanthocyanins                 0.41     3.58    1.590899    0.572359
Color intensity                 1.28    13.00    5.058090    2.318286
Hue                             0.4

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - Labels are already encoded as 0, 1, and 2.

X = df[feature_names].to_numpy(dtype=np.float32)
y = df["Class"].to_numpy(dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First encoded labels:", y[:10])


X shape: (178, 13)
y shape: (178,)
First encoded labels: [0 0 0 0 0 0 0 0 0 0]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42.
# Stratification preserves the class distribution in both splits.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("Training class counts:", np.bincount(y_train))
print("Test class counts:", np.bincount(y_test))


Training samples: 124
Test samples: 54
Training class counts: [41 50 33]
Test class counts: [18 21 15]


In [8]:
# Step 3: Use StandardScaler to normalize the features.
# - Fit on X_train and transform both X_train and X_test.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print("Scaled training feature means, first 5:", np.round(X_train_scaled.mean(axis=0)[:5], 4))
print("Scaled training feature stds, first 5:", np.round(X_train_scaled.std(axis=0)[:5], 4))


Scaled training feature means, first 5: [ 0. -0.  0. -0.  0.]
Scaled training feature stds, first 5: [1. 1. 1. 1. 1.]


In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes).astype(np.float32)
y_test_cat = to_categorical(y_test, num_classes=num_classes).astype(np.float32)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)
print("First encoded label:", y_train[0], "->", y_train_cat[0])


y_train_cat shape: (124, 3)
y_test_cat shape: (54, 3)
First encoded label: 0 -> [1. 0. 0.]


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

model = Sequential(
    [
        Dense(64, activation="relu", input_shape=(num_features,), name="dense_64"),
        Dense(32, activation="relu", name="dense_32"),
        Dense(num_classes, activation="softmax", name="wine_class"),
    ],
    name="wine_baseline_dnn",
)

model.summary()


Model: "wine_baseline_dnn"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense_64 (Dense)            (None, 64)                896       


 dense_32 (Dense)            (None, 32)                2080      


 wine_class (Dense)          (None, 3)                 99        


Total params: 3075 (12.01 KB)


Trainable params: 3075 (12.01 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric.
# - Train for 20 epochs with batch_size=8 and validation_split=0.2.

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=0,
)

train_loss, train_acc = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
val_acc = history.history["val_accuracy"][-1]

print(f"Final training accuracy: {train_acc:.4f}")
print(f"Final validation accuracy: {val_acc:.4f}")


Final training accuracy: 0.9919
Final validation accuracy: 0.9600


In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

base_test_loss, base_test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
base_probs = model.predict(X_test_scaled, verbose=0)
base_pred = np.argmax(base_probs, axis=1)

print(f"Baseline training accuracy: {train_acc:.4f}")
print(f"Baseline test accuracy: {base_test_acc:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, base_pred, target_names=class_names, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, base_pred))


Baseline training accuracy: 0.9919
Baseline test accuracy: 0.9815

Classification report:

              precision    recall  f1-score   support

  Cultivar 1       0.95      1.00      0.97        18
  Cultivar 2       1.00      0.95      0.98        21
  Cultivar 3       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite".
# - Print the file size in kilobytes.

converter = tf.lite.TFLiteConverter.from_keras_model(model)
base_tflite_model = convert_tflite(converter)
base_tflite_path = write_tflite_model(base_tflite_model, "model_base.tflite")
base_size_kb = file_size_kb("model_base.tflite")

results["baseline_float32"] = {
    "size_kb": base_size_kb,
    "accuracy": base_test_acc,
    "filename": str(base_tflite_path),
}

print("Saved baseline model to:", base_tflite_path)
print(f"Float32 TFLite model size: {base_size_kb:.2f} KB")


Saved baseline model to: /mnt/e/EE446/HW1/model_base.tflite
Float32 TFLite model size: 14.21 KB


## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def run_tflite_classifier(filename, X_eval):
    """Run a TFLite classifier on X_eval and return predicted class indices."""
    path = artifact_path(filename)
    interpreter = tf.lite.Interpreter(model_path=str(path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_dtype = input_details["dtype"]
    output_dtype = output_details["dtype"]

    predictions = []
    for sample in X_eval.astype(np.float32):
        input_data = sample.reshape(1, -1).astype(np.float32)

        if np.issubdtype(input_dtype, np.integer):
            input_scale, input_zero_point = input_details["quantization"]
            if input_scale == 0:
                raise ValueError("Quantized input tensor has zero scale.")
            input_data = np.round(input_data / input_scale + input_zero_point)
            input_limits = np.iinfo(input_dtype)
            input_data = np.clip(input_data, input_limits.min, input_limits.max).astype(input_dtype)
        else:
            input_data = input_data.astype(input_dtype)

        interpreter.set_tensor(input_details["index"], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details["index"])[0]

        if np.issubdtype(output_dtype, np.integer):
            output_scale, output_zero_point = output_details["quantization"]
            output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale

        predictions.append(int(np.argmax(output_data)))

    return np.array(predictions, dtype=np.int64)


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename, X_reference=None):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    X_reference : np.ndarray, optional
        Representative training samples for full integer quantization.
    """

    if X_reference is None:
        X_reference = X_train_scaled

    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == "int8":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_reference)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == "float16":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == "dynamic":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = convert_tflite(converter)
    path = write_tflite_model(tflite_model, filename)

    # Step 3: Run TFLite inference.
    y_pred = run_tflite_classifier(path, X_test)
    y_true = np.argmax(y_test_cat, axis=1)
    accuracy = accuracy_score(y_true, y_pred)

    # Step 4: Report results.
    size_kb = file_size_kb(path)
    print(f"\n{quant_type.upper()} TFLite model: {path}")
    print(f"{quant_type.upper()} TFLite model size: {size_kb:.2f} KB")
    print(f"{quant_type.upper()} TFLite test accuracy: {accuracy:.4f}")
    print("\nClassification report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

    return {
        "size_kb": size_kb,
        "accuracy": accuracy,
        "filename": str(path),
        "y_pred": y_pred,
    }


In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quant_results = {}
quant_results["int8"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    "int8",
    "model_int8.tflite",
    X_reference=X_train_scaled,
)
quant_results["float16"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    "float16",
    "model_float16.tflite",
)
quant_results["dynamic"] = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    "dynamic",
    "model_dynamic.tflite",
)

for key, value in quant_results.items():
    results[f"baseline_{key}"] = value


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.



INT8 TFLite model: /mnt/e/EE446/HW1/model_int8.tflite
INT8 TFLite model size: 5.88 KB
INT8 TFLite test accuracy: 0.9815

Classification report:

              precision    recall  f1-score   support

  Cultivar 1       0.95      1.00      0.97        18
  Cultivar 2       1.00      0.95      0.98        21
  Cultivar 3       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]



FLOAT16 TFLite model: /mnt/e/EE446/HW1/model_float16.tflite
FLOAT16 TFLite model size: 9.14 KB
FLOAT16 TFLite test accuracy: 0.9815

Classification report:

              precision    recall  f1-score   support

  Cultivar 1       0.95      1.00      0.97        18
  Cultivar 2       1.00      0.95      0.98        21
  Cultivar 3       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]



DYNAMIC TFLite model: /mnt/e/EE446/HW1/model_dynamic.tflite
DYNAMIC TFLite model size: 8.31 KB
DYNAMIC TFLite test accuracy: 0.9815

Classification report:

              precision    recall  f1-score   support

  Cultivar 1       0.95      1.00      0.97        18
  Cultivar 2       1.00      0.95      0.98        21
  Cultivar 3       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay.
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

pruning_epochs = 10
pruning_batch_size = 8
pruning_validation_split = 0.2
pruning_train_samples = int(X_train_scaled.shape[0] * (1 - pruning_validation_split))
end_step = int(np.ceil(pruning_train_samples / pruning_batch_size) * pruning_epochs)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step,
)

print("Pruning train samples:", pruning_train_samples)
print("Pruning end_step:", end_step)


Pruning train samples: 99
Pruning end_step: 130


In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude().

tf.keras.utils.set_random_seed(142)

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential(
    [
        prune_low_magnitude(
            Dense(64, activation="relu", input_shape=(num_features,), name="pruned_dense_64"),
            pruning_schedule=pruning_schedule,
        ),
        prune_low_magnitude(
            Dense(32, activation="relu", name="pruned_dense_32"),
            pruning_schedule=pruning_schedule,
        ),
        prune_low_magnitude(
            Dense(num_classes, activation="softmax", name="pruned_wine_class"),
            pruning_schedule=pruning_schedule,
        ),
    ],
    name="wine_pruned_dnn",
)

pruned_model.summary()


Model: "wine_pruned_dnn"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 prune_low_magnitude_pruned  (None, 64)                1730      


 _dense_64 (PruneLowMagnitu                                      


...

 tude)                                                           


Total params: 6057 (23.67 KB)


Trainable params: 3075 (12.01 KB)


Non-trainable params: 2982 (11.66 KB)


_________________________________________________________________


In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy.
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list.

pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

pruned_history = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=pruning_epochs,
    batch_size=pruning_batch_size,
    validation_split=pruning_validation_split,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=0,
)

pruned_train_loss, pruned_train_acc = pruned_model.evaluate(X_train_scaled, y_train_cat, verbose=0)
print(f"Pruned wrapped model training accuracy: {pruned_train_acc:.4f}")
print(f"Pruned wrapped model validation accuracy: {pruned_history.history['val_accuracy'][-1]:.4f}")


Pruned wrapped model training accuracy: 0.9919
Pruned wrapped model validation accuracy: 0.9600


In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.
#
# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
stripped_pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)


def dense_weight_sparsity(model_to_check):
    zero_count = 0
    total_count = 0
    for weight in model_to_check.get_weights():
        if weight.ndim >= 2:
            total_count += weight.size
            zero_count += np.sum(np.isclose(weight, 0.0))
    return zero_count / total_count if total_count else 0.0

pruned_sparsity = dense_weight_sparsity(stripped_pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
pruned_tflite_model = convert_tflite(converter)
pruned_tflite_path = write_tflite_model(pruned_tflite_model, "model_pruned.tflite")
pruned_size_kb = file_size_kb("model_pruned.tflite")

print("Saved pruned model to:", pruned_tflite_path)
print(f"Dense weight sparsity after stripping: {pruned_sparsity:.2%}")
print(f"Pruned float32 TFLite model size: {pruned_size_kb:.2f} KB")


Saved pruned model to: /mnt/e/EE446/HW1/model_pruned.tflite
Dense weight sparsity after stripping: 69.76%
Pruned float32 TFLite model size: 14.33 KB


In [20]:
# Step 5: Evaluate using the stripped model.
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix.

pruned_test_loss, pruned_test_acc = stripped_pruned_model.evaluate(X_test_scaled, y_test_cat, verbose=0)
pruned_probs = stripped_pruned_model.predict(X_test_scaled, verbose=0)
pruned_pred = np.argmax(pruned_probs, axis=1)

results["pruned_float32"] = {
    "size_kb": pruned_size_kb,
    "accuracy": pruned_test_acc,
    "filename": str(pruned_tflite_path),
    "sparsity": pruned_sparsity,
}

print(f"Pruned model test accuracy: {pruned_test_acc:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, pruned_pred, target_names=class_names, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pruned_pred))


Pruned model test accuracy: 0.9630

Classification report:

              precision    recall  f1-score   support

  Cultivar 1       0.95      1.00      0.97        18
  Cultivar 2       1.00      0.90      0.95        21
  Cultivar 3       0.94      1.00      0.97        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion matrix:
 [[18  0  0]
 [ 1 19  1]
 [ 0  0 15]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

tf.keras.utils.set_random_seed(105)

student_model = Sequential(
    [
        Dense(32, activation="relu", input_shape=(num_features,), name="student_dense_32"),
        Dense(16, activation="relu", name="student_dense_16"),
        Dense(num_classes, activation="softmax", name="student_wine_class"),
    ],
    name="wine_student_dnn",
)

student_model.summary()


Model: "wine_student_dnn"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 student_dense_32 (Dense)    (None, 32)                448       


 student_dense_16 (Dense)    (None, 16)                528       


 student_wine_class (Dense)  (None, 3)                 51        


Total params: 1027 (4.01 KB)


Trainable params: 1027 (4.01 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels.

teacher_preds_soft = model.predict(X_train_scaled, verbose=0).astype(np.float32)

print("Teacher soft labels shape:", teacher_preds_soft.shape)
print("First teacher soft label:", np.round(teacher_preds_soft[0], 4))
print("Corresponding hard label:", y_train_cat[0])


Teacher soft labels shape: (124, 3)
First teacher soft label: [9.987e-01 8.000e-04 5.000e-04]
Corresponding hard label: [1. 0. 0.]


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5
#
# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels.

alpha = 0.5
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1).astype(np.float32)

print("Combined distillation label shape:", y_train_combined.shape)


def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1 - alpha) * soft_loss


Combined distillation label shape: (124, 6)


In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss.
# - Train for 10 epochs, batch_size=8, validation_split=0.2.
# Accuracy is computed manually because y_train_combined contains both hard and soft labels.

student_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=distillation_loss,
)

student_history = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=0,
)

student_train_pred = np.argmax(student_model.predict(X_train_scaled, verbose=0), axis=1)
student_train_acc = accuracy_score(y_train, student_train_pred)

print(f"Student distillation training accuracy: {student_train_acc:.4f}")
print(f"Student final training distillation loss: {student_history.history['loss'][-1]:.4f}")


Student distillation training accuracy: 0.9919
Student final training distillation loss: 0.0342


In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
student_tflite_model = convert_tflite(converter)
student_tflite_path = write_tflite_model(student_tflite_model, "model_kd.tflite")
student_size_kb = file_size_kb("model_kd.tflite")

print("Saved knowledge-distilled student model to:", student_tflite_path)
print(f"KD student float32 TFLite model size: {student_size_kb:.2f} KB")


Saved knowledge-distilled student model to: /mnt/e/EE446/HW1/model_kd.tflite
KD student float32 TFLite model size: 6.33 KB


In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled.
# - Print classification_report and confusion_matrix.

student_probs = student_model.predict(X_test_scaled, verbose=0)
student_pred = np.argmax(student_probs, axis=1)
student_acc = accuracy_score(y_test, student_pred)

results["kd_student_float32"] = {
    "size_kb": student_size_kb,
    "accuracy": student_acc,
    "filename": str(student_tflite_path),
}

print(f"KD student test accuracy: {student_acc:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, student_pred, target_names=class_names, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, student_pred))


KD student test accuracy: 0.9815

Classification report:

              precision    recall  f1-score   support

  Cultivar 1       0.95      1.00      0.97        18
  Cultivar 2       1.00      0.95      0.98        21
  Cultivar 3       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
# Proposed strategy: combine architectural compression with full integer quantization.
# The Wine task is small and well separated, so a one-hidden-layer tiny student can
# preserve high accuracy while reducing the number of stored parameters sharply.

tf.keras.utils.set_random_seed(266)

tiny_student_model = Sequential(
    [
        Dense(16, activation="relu", input_shape=(num_features,), name="tiny_dense_16"),
        Dense(num_classes, activation="softmax", name="tiny_wine_class"),
    ],
    name="wine_tiny_student_dnn",
)

tiny_teacher_preds = model.predict(X_train_scaled, verbose=0).astype(np.float32)
tiny_train_combined = np.concatenate([y_train_cat, tiny_teacher_preds], axis=1).astype(np.float32)

tiny_alpha = 0.8


def tiny_distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return tiny_alpha * hard_loss + (1 - tiny_alpha) * soft_loss


tiny_student_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tiny_distillation_loss,
)

tiny_history = tiny_student_model.fit(
    X_train_scaled,
    tiny_train_combined,
    epochs=60,
    batch_size=8,
    validation_split=0.2,
    verbose=0,
)

tiny_float_pred = np.argmax(tiny_student_model.predict(X_test_scaled, verbose=0), axis=1)
tiny_float_acc = accuracy_score(y_test, tiny_float_pred)
print(f"Tiny KD float32 test accuracy before quantization: {tiny_float_acc:.4f}")
print(f"Tiny KD final training distillation loss: {tiny_history.history['loss'][-1]:.4f}")

tiny_int8_result = quantize_and_evaluate(
    tiny_student_model,
    X_test_scaled,
    y_test_cat,
    "int8",
    "model_tiny_kd_int8.tflite",
    X_reference=X_train_scaled,
)
results["tiny_kd_int8"] = tiny_int8_result

summary_df = (
    pd.DataFrame(
        [
            {"model": name, "size_kb": info["size_kb"], "accuracy": info["accuracy"], "filename": info["filename"]}
            for name, info in results.items()
        ]
    )
    .sort_values("size_kb")
    .reset_index(drop=True)
)

print("\nModel size and accuracy comparison:\n")
print(summary_df.to_string(index=False, formatters={"size_kb": "{:.2f}".format, "accuracy": "{:.4f}".format}))

smallest_before_part_e = min(
    (item for item in results.items() if item[0] != "tiny_kd_int8"),
    key=lambda item: item[1]["size_kb"],
)
smallest_after_part_e = min(results.items(), key=lambda item: item[1]["size_kb"])

print("\nSmallest model before Part (e):", smallest_before_part_e[0], f"({smallest_before_part_e[1]['size_kb']:.2f} KB)")
print("Smallest model after Part (e):", smallest_after_part_e[0], f"({smallest_after_part_e[1]['size_kb']:.2f} KB)")

if smallest_after_part_e[0] == "tiny_kd_int8":
    print(
        "\nPart (e) result: The tiny KD model with full integer quantization reduced size further. "
        "The biggest contributor was replacing the 64/32 hidden-layer baseline with a single 16-unit hidden layer, "
        "then storing weights and activations as int8."
    )
else:
    print(
        "\nPart (e) result: The attempted tiny KD + int8 model did not beat the previous smallest model. "
        "For this network, fixed TFLite metadata overhead and the minimum operators needed for inference limit further reduction."
    )


Tiny KD float32 test accuracy before quantization: 0.9815
Tiny KD final training distillation loss: 0.0160



INT8 TFLite model: /mnt/e/EE446/HW1/model_tiny_kd_int8.tflite
INT8 TFLite model size: 2.47 KB
INT8 TFLite test accuracy: 0.9815

Classification report:

              precision    recall  f1-score   support

  Cultivar 1       1.00      1.00      1.00        18
  Cultivar 2       1.00      0.95      0.98        21
  Cultivar 3       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]

Model size and accuracy comparison:

             model size_kb accuracy                                   filename
      tiny_kd_int8    2.47   0.9815 /mnt/e/EE446/HW1/model_tiny_kd_int8.tflite
     baseline_int8    5.88   0.9815         /mnt/e/EE446/HW1/model_int8.tflite
kd_student_float32    6.33   0.9815           /mnt/e/EE446/HW1/model_kd.tflite
  baseline_dynamic    8.31   0.9815      /mnt/e/EE446/H

fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
